In [2]:
!pip -q install streamlit
!npm -q install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 84.8 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 106.3 MB/s eta 0:00:0000:010:01
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 1s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [3]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import glob
from sklearn.metrics import cohen_kappa_score, accuracy_score


# 1. PAGE CONFIGURATION & CUSTOM STYLING

st.set_page_config(
    page_title="Clinical AI Pipeline Auditor",
    page_icon="🛡️",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
    .stage-card {
        border-left: 4px solid #3b82f6;
        padding: 12px 16px;
        background-color: rgba(59, 130, 246, 0.05);
        border-radius: 0 8px 8px 0;
        margin-bottom: 16px;
    }
    .status-badge-pass { background-color: rgba(16, 185, 129, 0.2); color: #10b981; padding: 4px 8px; border-radius: 4px; font-weight: 600; font-size: 0.85rem;}
    .status-badge-fail { background-color: rgba(239, 68, 68, 0.2); color: #ef4444; padding: 4px 8px; border-radius: 4px; font-weight: 600; font-size: 0.85rem;}
</style>
""", unsafe_allow_html=True)

#  DATA LOADER
@st.cache_data
def load_pipeline_data():
    # Prioritize local directories per the GitHub README, followed by Kaggle paths
    search_dirs = ["output/", "./", "/kaggle/input/**/", "/kaggle/working/"]
    
    def find_file(pattern):
        for s_dir in search_dirs:
            # Check direct path first, then recursive
            direct_path = os.path.join(s_dir, pattern)
            if os.path.exists(direct_path):
                return direct_path
            
            matches = glob.glob(os.path.join(s_dir, "**", pattern), recursive=True)
            if matches: 
                return matches[0]
        return None

    data = {}
    
    p_human = find_file("Manual_Annotation.csv")
    p_s1_llm = find_file("stage1_llama_extraction.parquet")
    p_cf_race = find_file("stage2_cf_race_output.parquet")
    p_cf_sex = find_file("stage2_cf_sex_output.parquet")
    
    if p_s1_llm and p_cf_race and p_cf_sex:
        try:
            df_llm = pd.read_parquet(p_s1_llm)
            df_cfr = pd.read_parquet(p_cf_race)
            df_cfs = pd.read_parquet(p_cf_sex)
            
            df_master = df_llm[["hadm_id", "comorbidity_burden_score"]].copy()
            df_master.rename(columns={"comorbidity_burden_score": "factual_score"}, inplace=True)
            
            r_score_col = [c for c in df_cfr.columns if "score" in c or "burden" in c][0]
            r_sim_col = [c for c in df_cfr.columns if "sim" in c or "cosine" in c][0]
            s_score_col = [c for c in df_cfs.columns if "score" in c or "burden" in c][0]
            s_sim_col = [c for c in df_cfs.columns if "sim" in c or "cosine" in c][0]
            
            df_master["cf_race_score"] = df_cfr[r_score_col]
            df_master["similarity_race"] = df_cfr[r_sim_col]
            df_master["cf_sex_score"] = df_cfs[s_score_col]
            df_master["similarity_sex"] = df_cfs[s_sim_col]
            
            data["cf_data"] = df_master
        except Exception:
            data["cf_data"] = None
    else:
        data["cf_data"] = None

    p_bench = find_file("fairness_full_results.parquet")
    p_l2 = find_file("level2_vae_full_matrix.parquet")
    p_mppd = find_file("mppd_results.parquet")
    p_shap = find_file("stage2_shap.parquet")

    data["benchmark"] = pd.read_parquet(p_bench) if p_bench else None
    data["level2"] = pd.read_parquet(p_l2) if p_l2 else None
    data["mppd"] = pd.read_parquet(p_mppd) if p_mppd else None
    data["shap"] = pd.read_parquet(p_shap) if p_shap else None

    # FALLBACK GENERATORS 
    if data["cf_data"] is None:
        np.random.seed(42)
        n = 400
        f_scores = np.random.randint(0, 10, n)
        data["cf_data"] = pd.DataFrame({
            "hadm_id": range(20000000, 20000000 + n),
            "factual_score": f_scores,
            "cf_race_score": np.clip(f_scores + np.random.choice([-3, -1, 0, 0, 1, 3], n), 0, 10),
            "cf_sex_score": np.clip(f_scores + np.random.choice([-2, 0, 0, 0, 2], n), 0, 10),
            "similarity_race": np.clip(np.random.normal(0.999, 0.001, n), 0.92, 1.0),
            "similarity_sex": np.clip(np.random.normal(0.998, 0.0015, n), 0.90, 1.0),
        })
        
    if data["benchmark"] is None:
        data["benchmark"] = pd.DataFrame([
            {"method": "Baseline", "auroc": 0.7017, "dp_ratio": 0.371, "cfvr": 0.0208, "bmr": 0.0000},
            {"method": "Reweighting", "auroc": 0.6909, "dp_ratio": 0.603, "cfvr": 0.0744, "bmr": 2.5769},
            {"method": "Equalized Odds", "auroc": np.nan, "dp_ratio": 0.910, "cfvr": 0.3474, "bmr": 15.7019},
            {"method": "Adversarial Debiasing", "auroc": 0.6838, "dp_ratio": 0.474, "cfvr": 0.0171, "bmr": -0.1779},
            {"method": "CDA", "auroc": 0.6960, "dp_ratio": 0.435, "cfvr": 0.0000, "bmr": -1.0000},
            {"method": "Path-Aware CDA", "auroc": 0.6921, "dp_ratio": 0.429, "cfvr": 0.0039, "bmr": -0.8125},
            {"method": "DoWhy SCM", "auroc": 0.6961, "dp_ratio": 0.429, "cfvr": 0.0035, "bmr": -0.8317},
            {"method": "Pipeline-Aware Hybrid", "auroc": 0.7017, "dp_ratio": 0.359, "cfvr": 0.0219, "bmr": 0.0529}
        ])

    return data

pipeline_data = load_pipeline_data()

# Add a professional display ID for the UI
if pipeline_data["cf_data"] is not None:
    pipeline_data["cf_data"]["display_id"] = "HADM-" + pipeline_data["cf_data"]["hadm_id"].astype(str)

# 3. SIDEBAR CONTROLS

with st.sidebar:
    st.title("🎛️ Pipeline Audit Controls")
    
    selected_axis = st.radio("Demographic Axis", ["Race", "Sex"], index=0)
    st.divider()
    
    sbert_threshold = st.slider(
        "SBERT Preservation Gate",
        min_value=0.85, max_value=1.00, value=0.95, step=0.01,
        help="Filters out perturbed notes that lost their clinical meaning."
    )
    st.divider()
    
    methods = pipeline_data["benchmark"]["method"].tolist()
    selected_method = st.selectbox("Compare Mitigation Method", methods, index=5 if len(methods) > 5 else 0)

# 4. MAIN HEADER & KPIs
st.title("🛡️ Clinical AI Safety & Bias Migration Explorer")
st.markdown("Auditing multi-stage bias propagation and LLM token instability across healthcare pipelines.")

bench_df = pipeline_data["benchmark"]
sel_row = bench_df[bench_df["method"] == selected_method].iloc[0]
base_row = bench_df[bench_df["method"].str.lower() == "baseline"].iloc[0]

kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1.metric("Validation Cohort", "60,391 admissions", "MIMIC-IV")
kpi2.metric("Demographic Parity", f"{sel_row['dp_ratio']:.3f}", f"{sel_row['dp_ratio'] - base_row['dp_ratio']:+.3f} vs Base", delta_color="normal")
kpi3.metric("CFVR Violation Rate", f"{sel_row['cfvr']:.4f}", f"{sel_row['cfvr'] - base_row['cfvr']:+.4f} vs Base", delta_color="inverse")
kpi4.metric("Bias Migration (BMR)", f"{sel_row['bmr']:+.2f}", "Optimal: < 0.0" if sel_row['bmr'] < 0 else "Amplified", delta_color="inverse" if sel_row['bmr'] > 0 else "normal")

st.divider()

# 5. TABS
tab1, tab2, tab3, tab4 = st.tabs([
    "Stage 1: NLP Extraction Audit",
    "Stages 2 & 3: Migrations",
    "Micro Bias Trace (Patient Inspector)",
    "EU AI Act Conformity"
])

# TAB 1: NLP EXTRACTION 
with tab1:
    st.subheader("Human-in-the-Loop & SBERT Semantic Gating")
    c1, c2 = st.columns([1, 1.2])
    
    with c1:
        st.markdown("**Extraction Reliability vs Human Annotators ($N=41$)**")
        col_a, col_b = st.columns(2)
        col_a.metric("Comorbidity QW Kappa (κ)", "0.766", "Substantial")
        col_b.metric("SDOH Exact Match", "92.7%", "Spec: 97.4%")
        col_a.metric("Psych Complexity Match", "75.6%")
        col_b.metric("Discharge Indicators (MAE)", "1.27")

    with c2:
        cf_df = pipeline_data["cf_data"]
        sim_col = "similarity_race" if selected_axis == "Race" else "similarity_sex"
        cf_score_col = "cf_race_score" if selected_axis == "Race" else "cf_sex_score"
        
        # Apply the Validity Mask
        cf_df["Valid_SBERT"] = cf_df[sim_col] >= sbert_threshold
        
        fig_sbert = px.histogram(cf_df, x=sim_col, nbins=40, color_discrete_sequence=["#0284c7"], title="SBERT Cosine Similarity")
        fig_sbert.add_vline(x=sbert_threshold, line_dash="dash", line_color="#ef4444", annotation_text="Gate")
        fig_sbert.update_layout(height=280, margin=dict(l=10, r=10, t=30, b=10))
        st.plotly_chart(fig_sbert, use_container_width=True)

    st.divider()

    # PERTURBATION INSTABILITY SECTION
    st.subheader("The Perturbation Instability Phenomenon")
    st.markdown(r"A perfectly stable LLM extraction evaluates identically regardless of demographic swaps (falling exactly on the $y=x$ line). Deviations from this line reveal inherent stochastic noise triggered strictly by the attention cascade.")
    
    # Interactive Toggle for Invalid Notes
    filter_invalid = st.toggle("Filter out notes that failed the SBERT Semantic Gate", value=True)
    plot_df = cf_df[cf_df["Valid_SBERT"]] if filter_invalid else cf_df
    divergence = plot_df["factual_score"] - plot_df[cf_score_col]

    col_div1, col_div2 = st.columns(2)
    
    with col_div1:
        # 2D Scatter Plot
        fig_scatter = px.scatter(
            plot_df, 
            x="factual_score", 
            y=cf_score_col, 
            color="Valid_SBERT" if not filter_invalid else None,
            color_discrete_map={True: "#10b981", False: "#ef4444"},
            opacity=0.5,
            labels={"factual_score": "Factual LLM Score", cf_score_col: f"Counterfactual LLM Score ({selected_axis})"},
            title="Extraction Divergence Scatter"
        )
        # Add the line of perfect fairness (y=x) with slate gray color for dark mode visibility
        fig_scatter.add_shape(type="line", x0=0, y0=0, x1=10, y1=10, line=dict(color="#94a3b8", dash="dash"))
        fig_scatter.update_layout(height=350, margin=dict(l=10, r=10, t=30, b=10))
        st.plotly_chart(fig_scatter, use_container_width=True)

    with col_div2:
        # 1D Histogram of the Delta
        fig_hist = px.histogram(
            x=divergence, 
            nbins=15, 
            color_discrete_sequence=["#10b981" if filter_invalid else "#6366f1"], 
            labels={"x": r"Score Divergence ($\Delta f(x)$)"},
            title="Divergence Distribution Magnitude"
        )
        # Set zero-line to slate gray for dark mode visibility
        fig_hist.add_vline(x=0, line_dash="dash", line_color="#94a3b8", line_width=2)
        fig_hist.update_layout(height=350, margin=dict(l=10, r=10, t=30, b=10))
        st.plotly_chart(fig_hist, use_container_width=True)

# TAB 2: MODEL BENCHMARKS 
with tab2:
    st.subheader("Downstream Mitigation Benchmarks")
    disp_df = bench_df.copy()
    disp_df["auroc"] = disp_df["auroc"].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "N/A")
    st.dataframe(disp_df, use_container_width=True, hide_index=True)
    
    c3, c4 = st.columns(2)
    with c3:
        st.markdown("**Level 1 vs Level 2 (Latent VAE) CFVR**")
        l2_df = pipeline_data["level2"]
        if l2_df is not None:
            fig_l2 = go.Figure(data=[
                go.Bar(name="Level 1", x=l2_df["method"], y=l2_df["cfvr_l1_race"], marker_color="#6366f1"),
                go.Bar(name="Level 2 (VAE)", x=l2_df["method"], y=l2_df["cfvr_l2_adj_race"], marker_color="#f59e0b")
            ])
            fig_l2.update_layout(barmode="group", height=300, margin=dict(l=10, r=10, t=10, b=10))
            st.plotly_chart(fig_l2, use_container_width=True)
        else:
            st.info("Level 2 data loading... (Awaiting Parquet)")
            
    with c4:
        st.markdown("**MPPD: Real-Patient Validation Cross-Check**")
        mppd_df = pipeline_data["mppd"]
        if mppd_df is not None:
            if "race" in mppd_df.columns and "mppd" in mppd_df.columns:
                fig_mppd = px.bar(
                    mppd_df,
                    x="method",
                    y="mppd",
                    color="race",
                    barmode="group",
                    labels={
                        "mppd": "Mean Prediction Diff (MPPD)",
                        "method": "Method",
                        "race": "Demographic Group"
                    },
                    color_discrete_sequence=["#38bdf8", "#818cf8", "#f472b6", "#fb923c"]
                )
            else:
                fig_mppd = px.bar(
                    mppd_df,
                    x="method",
                    y=["Asian", "Black/African American", "Hispanic/Latino", "Other/Unknown"],
                    barmode="group",
                    color_discrete_sequence=["#38bdf8", "#818cf8", "#f472b6", "#fb923c"]
                )
            fig_mppd.update_layout(height=300, margin=dict(l=10, r=10, t=10, b=10), legend=dict(orientation="h", y=1.15))
            st.plotly_chart(fig_mppd, use_container_width=True)
        else:
            st.info("MPPD data loading... (Awaiting Parquet)")

# TAB 3: MICRO BIAS TRACE 
with tab3:
    st.subheader("Patient Encounter Inspector")
    cf_df = pipeline_data["cf_data"]
    
    # Query using the newly created display_id
    chosen_display_id = st.selectbox("Select Admission ID (`hadm_id`)", cf_df["display_id"].tolist())
    patient = cf_df[cf_df["display_id"] == chosen_display_id].iloc[0]
    
    st.markdown("""<div class="stage-card"><h4>Stage 1: LLM Extraction</h4></div>""", unsafe_allow_html=True)
    m1, m2, m3 = st.columns(3)
    m1.metric("Factual Score", f"{patient['factual_score']} / 10")
    
    cfs = patient['cf_race_score'] if selected_axis == "Race" else patient['cf_sex_score']
    sim = patient['similarity_race'] if selected_axis == "Race" else patient['similarity_sex']
    
    m2.metric(f"Counterfactual Score ({selected_axis})", f"{cfs} / 10", f"{cfs - patient['factual_score']:+g} Shift")
    pass_html = '<span class="status-badge-pass">PASSED</span>' if sim >= sbert_threshold else '<span class="status-badge-fail">REJECTED</span>'
    m3.markdown(f"**SBERT Gate:** {sim:.3f}<br>{pass_html}", unsafe_allow_html=True)

    st.markdown("""<div class="stage-card"><h4>Stage 2 & 3: Risk Calibration & Enrollment</h4></div>""", unsafe_allow_html=True)
    p_fac = np.clip(0.15 + (patient['factual_score'] * 0.04), 0.05, 0.85)
    p_cf = np.clip(0.15 + (cfs * 0.04), 0.05, 0.85)
    
    r1, r2, r3 = st.columns(3)
    r1.metric("Factual Calibrated Risk P(Y=1)", f"{p_fac:.3f}")
    r2.metric("Counterfactual Risk P(Y'=1)", f"{p_cf:.3f}", f"{p_cf - p_fac:+.3f}")
    
    fac_enr = p_fac >= 0.402
    cf_enr = p_cf >= 0.402
    
    r3.markdown(f"**Factual:** {'✅ Enrolled' if fac_enr else '❌ Denied'}<br>**Counterfactual:** {'✅ Enrolled' if cf_enr else '❌ Denied'}", unsafe_allow_html=True)

    if fac_enr != cf_enr:
        st.error("⚠️ **CFVR VIOLATION DETECTED:** The enrollment decision flipped strictly due to demographic token perturbation.")

# TAB 4: EU AI ACT 
with tab4:
    st.subheader("EU AI Act Conformity Mapping (High-Risk Healthcare Category)")
    e1, e2 = st.columns(2)
    with e1:
        st.markdown(r"""
        * **Article 9 (Risk Management):** Tracking $\Delta f(x)$ divergence.
        * **Article 10 (Data Governance):** Complete-case exclusion tracking.
        * **Article 13 (Transparency):** SHAP attributions and ECE calibration checks.
        * **Article 14 (Human Oversight):** Expert validation ($\kappa = 0.766$).
        * **Article 15 (Accuracy & Robustness):** SBERT gating isolates fragility.
        """)
    with e2:
        st.warning("""
        **Compliance Alert: Risk of False Assurance**
        
        Relying strictly on Level 1 Counterfactual Data Augmentation (CDA) yields a CFVR of 0.0000. Real-patient cross-checking (MPPD) and Level 2 latent analysis reveal this is an artifact of the evaluation metric mirroring the training assumption, not a solved bias issue.
        """)

Writing app.py


In [4]:
import urllib
import time

# Print the Endpoint IP required for the localtunnel security gate
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")

print(f"YOUR LOCALTUNNEL PASSWORD (ENDPOINT IP) IS: {ip}")
print("Copy this IP address and paste it into the tunnel authentication page.")

# Launch Streamlit 
!streamlit run app.py & npx localtunnel --port 8501

YOUR LOCALTUNNEL PASSWORD (ENDPOINT IP) IS: 34.61.233.28
Copy this IP address and paste it into the tunnel authentication page.
⠙your url is: https://all-clowns-drive.loca.lt


2026-08-30 20:24:27.106 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.19.2.2:8501
  External URL: http://34.61.233.28:8501

^C
  Stopping...
